# Densification Analysis of Stochastic Quantum Trajectories

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3D
import matplotlib.animation as animation

from numba import njit
from numba import complex128 as ncomplex

%matplotlib widget

## Function

In [ ]:
# ── Bloch sphere helpers ───────────────────────────────────────────────────

sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

def bloch_coords(rho):
    """Bloch vector (x,y,z) from a 2x2 density matrix."""
    x = np.real(np.trace(rho @ sigma_x))
    y = np.real(np.trace(rho @ sigma_y))
    z = np.real(np.trace(rho @ sigma_z))
    return np.array([x, y, z])

@njit
def NJIT_bloch_coords(rho):
    x = np.real(np.trace(rho @ np.array([[0, 1], [1, 0]], dtype=ncomplex)))
    y = np.real(np.trace(rho @ np.array([[0, -1j], [1j, 0]], dtype=ncomplex)))
    z = np.real(np.trace(rho @ np.array([[1, 0], [0, -1]], dtype=ncomplex)))
    return x, y, z

@njit
def NJIT_vectors_inCartesian_coords(many_rho_trjs, t_idx):
    """Estrae i vettori di Bloch di tutte le traiettorie all'indice temporale t_idx."""
    n_traj = many_rho_trjs.shape[0]
    vects = np.empty((3, n_traj))
    for i in range(n_traj):
        rho = many_rho_trjs[i, t_idx]
        x, y, z = NJIT_bloch_coords(rho)
        vects[0, i] = x
        vects[1, i] = y
        vects[2, i] = z
    return vects

@njit
def NJIT_angle_between_vectors(u, v):
    dot, nu, nv = 0.0, 0.0, 0.0
    for i in range(3):
        dot += u[i] * v[i]
        nu  += u[i] * u[i]
        nv  += v[i] * v[i]
    cos_t = dot / (np.sqrt(nu) * np.sqrt(nv))
    cos_t = max(-1.0, min(1.0, cos_t))
    return np.arccos(cos_t)

@njit
def NJIT_mean_angle_bruteforce(many_rho_trjs, t_idx, norm=np.pi):
    """Angolo medio (normalizzato) tra tutte le coppie di traiettorie al tempo t_idx."""
    av_angle = 0.0
    c = 0
    n = many_rho_trjs.shape[0]
    vects = NJIT_vectors_inCartesian_coords(many_rho_trjs, t_idx)
    for i in range(n):
        for j in range(i + 1, n):
            av_angle += NJIT_angle_between_vectors(vects[:, i], vects[:, j])
            c += 1
    return (av_angle / (c * norm)) if c > 0 else 0.0

@njit
def NJIT_syncr_measure_time(many_rho_trjs, norm=np.pi/2, minusone=True):
    """
    Misura di densificazione nel tempo.

    Parameters
    ----------
    many_rho_trjs : ndarray, shape (N_traj, N_time, 2, 2)
    norm          : normalizzazione angoli (default pi/2 → misura in [0,1])
    minusone      : se True restituisce 1-angolo (1=sincronizzate)
                    se False restituisce l'angolo medio (0=sincronizzate)

    Returns
    -------
    ndarray, shape (N_time,)
    """
    ntime = many_rho_trjs.shape[1]
    out   = np.zeros(ntime)
    for i in range(ntime):
        a = NJIT_mean_angle_bruteforce(many_rho_trjs, i, norm)
        out[i] = (1.0 - a) if minusone else a
    return out

## Non Markovian Measure -- Le ignoro ??

In [ ]:
# ── Misure non-Markoviane (da nonmarkov.py) ────────────────────────────────

def finite_diff_gradient(y, x):
    """Gradiente a differenze finite di y rispetto a x."""
    g = np.zeros_like(y)
    g[1:-1] = (y[2:] - y[:-2]) / (x[2:] - x[:-2])
    g[0]    = (y[1]  - y[0])   / (x[1]  - x[0])
    g[-1]   = (y[-1] - y[-2])  / (x[-1] - x[-2])
    return g

def get_positive_part(y):
    """Azzera i valori negativi di y."""
    return np.where(y > 0, y, 0.0)

def integrate_on_positive_part(x, y):
    """
    Integra la parte positiva di y su x.
    Restituisce (integrale_totale, integrale_cumulativo_in_tempo).
    """
    pos_y = get_positive_part(y)
    integral_t = np.cumsum(pos_y * np.diff(x, prepend=x[0]))
    return integral_t[-1], integral_t

def resynchronization_measure(syncr_meas, time_stepped):
    """
    Calcola la misura di risincronizzazione dalla misura di densificazione.
    Integra solo i tratti in cui la densificazione sta aumentando.
    """
    rate   = finite_diff_gradient(syncr_meas, time_stepped)
    pos    = get_positive_part(rate)
    total, in_time = integrate_on_positive_part(time_stepped, pos)
    return total, in_time, rate